<a href="https://colab.research.google.com/github/ze-16/Speech-training-agent/blob/main/IEMOCAP_processing_script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import gdown
import os
import tarfile

In [ ]:
def create_text_csv(wav,labels,sentences,path):
  df = pd.DataFrame(columns=['audio_name','utterance','emotion'])
  for audio_id in wav:
    utterance_list = sentences[audio_id]
    emotion_list = labels[audio_id]
    df2 = pd.DataFrame()
    df2['utterance'] = utterance_list
    df2['emotion'] = emotion_list
    df2['audio_name'] = [audio_id]*len(emotion_list)
    df = pd.concat([df,df2],ignore_index=True)
    df["Emotion"] = df["Emotion"].replace(4, 0)
    df = df[df["Emotion"] < 4]
  df.to_csv(path,index=False)

In [ ]:
tar_path = "/content/drive/MyDrive/iemocap_data/IEMOCAP_full_release_withoutVideos (1).tar.gz"
wav_output = "/content/training_data/wavs"
Label_sentence_data = "/content/training_data/label_sentence_data.csv"

In [ ]:
os.makedirs(wav_output,exist_ok=True)

In [ ]:
labels_dic = {}
sentences_dic = {}
wav_ids = []

In [ ]:
emo_map = {
    'neu': 0,
    'exc': 1,
    'hap': 1,
    'sad': 2,
    'fru': 3,
    'dis': 3,
    'ang': 3
}

In [ ]:
with tarfile.open(tar_path) as t:
  for member in t.getmembers():
    if "/dialog/EmoEvaluation" in member.name and member.name.endswith(".txt"):
      f = t.extractfile(member)
      if f is not None:
        content = f.read().decode('utf-8')
        for l in content.split('\n'):
          parts = l.split('\t')
          if len(parts) >= 3 and parts[1].startswith('Ses'):
            wav_id = parts[1]
            emotion_label = parts[2]
            labels_dic[wav_id] = emo_map.get(emotion_label, 99)
    elif "dialog/transcriptions" in member.name and member.name.endswith(".txt"):
      f = t.extractfile(member)
      if f is not None:
        content = f.read().decode('utf-8')
        for l in content.split('\n'):
          parts = l.split(']:')
          wav_id = parts[0].split ('[')[0].strip()
          if wav_id.startswith('Ses'):
            sentence = parts[1].strip()
            sentences_dic[wav_id] = sentence

KeyboardInterrupt: 

In [ ]:
keys_to_remove = []
for lbl,lbl_emo in labels_dic.items():
  if lbl_emo == 99:
    keys_to_remove.append(lbl)
for key in keys_to_remove:
  labels_dic.pop(key)

In [ ]:
print(labels_dic)

In [ ]:
for i in labels_dic.keys():
  if i in sentences_dic.keys():
    wav_ids.append(i)

In [ ]:
with tarfile.open(tar_path) as t:
  for w in t.getmembers():
    if "sentences/wav" in w.name and w.name.endswith(".wav"):
      w_id = w.name.split('/')[-1].split('.wav')[0]
      if w_id in wav_ids:
        f = t.extractfile(w)
        if f is not None:
          wav_data = f.read()
          with open(os.path.join(wav_output,w_id+'.wav'),'wb') as wav_file:
            wav_file.write(wav_data)
